# sPCA component scores vs. barcode UMI depth

This notebook re-runs the production waggr scoring on three cell subsets, mirroring the
count-depth design already used for the GLM coefficients in
`01_glmGamPoi_interaction_countSubset_slurm.r`:

- **`top30`** — top 30% of cells by mean barcode score
- **`bottom30`** — bottom 30%
- **`random30`** — a score-agnostic random 30% (seed 0), the subsampling-noise comparator that
  `top30`/`bottom30` are read against, matching the single `random30` fit in
  `01_glmGamPoi_interaction_countSubset_slurm.r`
- **`full`** — all cells, to reproduce the production scores as a pipeline check


In [ ]:
# Repo-relative paths: inputs are read from imports_stable/, outputs are written to analysis_outs/
import os
from pathlib import Path
_here = Path.cwd().resolve()
REPO_DIR = str(next(p for p in [_here, *_here.parents] if (p / "imports_stable").is_dir()))
IMPORTS_DIR = f"{REPO_DIR}/imports_stable"
OUTS_DIR = f"{REPO_DIR}/analysis_outs/02_combinatorial_screen_signalseq_SIG13"
for _d in ["qc_barcode_counts"]:
    os.makedirs(f"{OUTS_DIR}/{_d}", exist_ok=True)

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import scanpy as sc
import decoupler as dc

import sys
sys.path.insert(0, f"{REPO_DIR}/functions")
import scanpy_custom as scc

OUT_DIR = OUTS_DIR + "/qc_barcode_counts"

## Import

Same alpha/filter (`0.1_alpha1.0`) as the production scoring, and the *existing* loadings
are reused rather than refit — the question is about scores, not about the components.


In [2]:
rna = sc.read_h5ad(IMPORTS_DIR + "/SIG13/scanpy_outs/SIG13_doublets_DSB7.h5ad")
rna

AnnData object with n_obs × n_vars = 349678 × 13050
    obs: 'replicate', 'lane', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'log10_genes_per_umi', 'total_counts_bc', 'log1p_total_counts_bc', 'feature_call_DSB5', 'num_umis_DSB5', 'num_features_DSB5', 'ligand_call_DSB5', 'group_call_DSB5', 'feature_call_DSB7', 'num_umis_DSB7', 'num_features_DSB7', 'ligand_call_DSB7', 'group_call_DSB7', 'feature_call_DSB10', 'num_umis_DSB10', 'num_features_DSB10', 'ligand_call_DSB10', 'group_call_DSB10', 'num_features_Cr', 'feature_call_Cr', 'num_umis_Cr', 'ligand_call_Cr', 'group_call_Cr', 'S_score', 'G2M_score', 'phase', 'replicate_feature_call_DSB7', 'ligand_call_round1_DSB7', 'ligand_call_round2_DSB7'
    var: '

In [3]:
spca_components = pd.read_csv(IMPORTS_DIR + "/SIG13/analysis_outs/spca/zscore_degs_allLigands_0.1_alpha1.0_sPCA_loadings.csv")
net = spca_components.rename(columns={'gene':'target',
                                      'spca_component':'source',
                                      'loading':'weight'})

# define genes used in scoring
comp_genes = net['target'].unique().tolist()
print(f"{len(comp_genes)} genes across {net['source'].nunique()} components")

6610 genes across 78 components


## Subset genes and scale

Scaling is done **once, on the full cell population, before any cell subsetting** — the
same choice made in `analysis/spca_null/01_spca_waggr_scoring_null.ipynb`. Every subset then
shares one scaling reference, so score differences between subsets reflect which cells were
selected rather than a re-centered gene space.


In [4]:
# subset and scale
rna_sub = rna[:, comp_genes].copy()
rna_sub.X = rna_sub.layers['log1p_norm']
sc.pp.scale(rna_sub)

del rna

/data1/rudenska/EYW/mamba_envs/scanpy_standard2/lib/python3.14/functools.py:982: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


## Per-cell mean barcode score

Same definition as the GLM count-subset script
(`01_glmGamPoi_interaction_countSubset_slurm.r`, "Subset to the top or bottom X% of cells"
block): `num_umis_DSB7` is a `"|"`-joined string of the round1 and round2 DSB scores, **not**
a total, so it is split before any numeric use.


In [5]:
num_umis = rna_sub.obs['num_umis_DSB7'].astype(str)
n_fields = num_umis.str.count(r'\|') + 1
print("barcodes per cell (n_fields in num_umis_DSB7):")
print(n_fields.value_counts().sort_index())

parts = num_umis.str.split('|', expand=True)
mean_bc = parts.iloc[:, :2].apply(pd.to_numeric, errors='coerce').mean(axis=1)

# this object should be the two-barcode "correct doublet" population used by the GLM;
# anything else cannot be assigned a round1/round2 mean and is dropped
usable = (n_fields == 2) & mean_bc.notna()
print(f"\nusable cells: {usable.sum()} / {len(usable)} ({(~usable).sum()} dropped)")

rna_sub.obs['mean_barcode_score'] = mean_bc
rna_sub.obs['mean_barcode_score_usable'] = usable.values

barcodes per cell (n_fields in num_umis_DSB7):
num_umis_DSB7
2    349678
Name: count, dtype: int64

usable cells: 349678 / 349678 (0 dropped)


## Define the subsets

**Grouping unit for the quantiles is `(ligand_call_DSB7, replicate)`** — the pseudobulk unit
itself — so every pseudobulk mean is taken over that unit's own top / bottom / random 30% of
cells.

Units too small to give a stable 30% pseudobulk mean are dropped from **all** subsets, so every
comparison stays paired on the same set of units.


In [6]:
obs = rna_sub.obs
obs['unit'] = obs['ligand_call_DSB7'].astype(str) + '__' + obs['replicate'].astype(str)

MIN_SUBSET_CELLS = 10
# ceil(0.3 * 34) = 11 >= MIN_SUBSET_CELLS, with a little headroom for ties at the quantile
MIN_UNIT_CELLS = 34

n_per_unit = obs.loc[obs['mean_barcode_score_usable'], 'unit'].value_counts()
valid_units = set(n_per_unit[n_per_unit >= MIN_UNIT_CELLS].index)
dropped_units = sorted(set(n_per_unit.index) - valid_units)

print(f"units kept: {len(valid_units)} / {len(n_per_unit)}")
print(f"units dropped (< {MIN_UNIT_CELLS} cells, so < {MIN_SUBSET_CELLS} in a 30% subset): {len(dropped_units)}")
if dropped_units:
    print(n_per_unit.loc[dropped_units].sort_values().to_string())

units kept: 1313 / 1320
units dropped (< 34 cells, so < 10 in a 30% subset): 7
unit
IL6_IFNB1__rep1      25
IL23_IL1A__rep1      26
IL23_OSM__rep1       29
IL23_IL9__rep1       31
IL23_IL9__rep2       31
IL23_TNFSF9__rep1    31
IL23_TNFSF4__rep1    32


In [7]:
base = obs['mean_barcode_score_usable'] & obs['unit'].isin(valid_units)
sub = obs.loc[base]

grp = sub.groupby('unit', observed=True)['mean_barcode_score']
q70 = grp.transform(lambda s: s.quantile(0.7)).reindex(obs.index)
q30 = grp.transform(lambda s: s.quantile(0.3)).reindex(obs.index)

def random_mask(seed):
    # score-agnostic random 30% of cells within each unit, matched in size to top30/bottom30
    rng = np.random.default_rng(seed)
    chosen = []
    for _, g in sub.groupby('unit', observed=True, sort=True):
        k = int(np.ceil(0.3 * len(g)))
        chosen.append(rng.choice(g.index.values, size=k, replace=False))
    m = pd.Series(False, index=obs.index)
    m.loc[np.concatenate(chosen)] = True
    return m

masks = {
    'full':     base.copy(),
    'top30':    base & (obs['mean_barcode_score'] >= q70),
    'bottom30': base & (obs['mean_barcode_score'] <= q30),
}
masks['random30'] = random_mask(0)

pd.Series({k: int(v.sum()) for k, v in masks.items()}, name='n_cells').to_frame()

,n_cells
full,349473
top30,105159
bottom30,105159
random30,105434


## Score every subset

`times=0` (no permutation p-values) and `tmin=5` match the production `waggr` call.


In [8]:
score_frames = []

for name, mask in masks.items():
    pb = sc.get.aggregate(rna_sub[mask.values], by=['ligand_call_DSB7', 'replicate'], func='mean')
    dc.mt.waggr(pb, net, tmin=5, layer='mean', times=0, verbose=False)

    sdf = pd.DataFrame(np.asarray(pb.obsm['score_waggr']),
                       index=pb.obs_names,
                       columns=pb.obsm['score_waggr'].columns)
    long = (sdf.rename_axis('unit_id')
               .reset_index()
               .melt(id_vars='unit_id', var_name='component', value_name='score'))
    long = long.join(pb.obs[['ligand_call_DSB7', 'replicate']].astype(str), on='unit_id')
    long['subset'] = name
    score_frames.append(long)

    print(f"{name}: {pb.n_obs} units x {sdf.shape[1]} components")

scores_long = pd.concat(score_frames, ignore_index=True)
scores_long.shape

full: 1313 units x 78 components
top30: 1313 units x 78 components
bottom30: 1313 units x 78 components
random30: 1313 units x 78 components


(409656, 6)

## Subset QC

Per `(subset, unit)` library-size and complexity summaries, consumed by
`04_spca_score_concordance.Rmd` to test whether a component-score difference between subsets is
explained by the library-size difference between them.

The barcode-depth/library-size relationship itself is characterised in
`qc_barcode_cutoff/01_barcode_calling_comparison.ipynb` (library-size section), not here: at the production DSB7 population mean
barcode score tracks `total_counts` at Spearman 0.29, and this exact top30/bottom30 split leaves the
top subset with a median 53% more total counts. So the table below is expected to show a depth gap —
what 05 tests is whether that gap accounts for the component-score shifts.


In [9]:
qc_frames = []
for name, mask in masks.items():
    agg = (obs.loc[mask.values]
              .groupby(['ligand_call_DSB7', 'replicate'], observed=True)
              .agg(n_cells=('mean_barcode_score', 'size'),
                   mean_barcode_score=('mean_barcode_score', 'mean'),
                   mean_total_counts=('total_counts', 'mean'),
                   mean_n_genes=('n_genes_by_counts', 'mean'),
                   mean_pct_mt=('pct_counts_mt', 'mean'))
              .reset_index())
    agg['subset'] = name
    qc_frames.append(agg)

subset_qc = pd.concat(qc_frames, ignore_index=True)

subset_qc.groupby('subset')[['n_cells', 'mean_barcode_score', 'mean_total_counts', 'mean_n_genes']].mean()

,n_cells,mean_barcode_score,mean_total_counts,mean_n_genes
subset,,,,
bottom30,80.090632,10.002098,6685.812012,3054.223598
full,266.163747,12.281130,8305.336914,3465.976983
random30,80.300076,12.282320,8298.791016,3464.364098
top30,80.090632,14.711333,10184.060547,3911.914986


## Pipeline check against the production scores

The `full` subset should reproduce
`analysis_outs/spca/degs_zscore_allLigands/zscore_degs_allLigands_0.1_alpha1.0_waggr_score.csv`
on the shared units. Anything below a near-perfect correlation means this notebook has diverged
from the production scoring and nothing downstream should be interpreted.


In [10]:
prod = pd.read_csv(IMPORTS_DIR + "/SIG13/analysis_outs/spca/degs_zscore_allLigands/zscore_degs_allLigands_0.1_alpha1.0_waggr_score.csv",
                   index_col=0)
prod_long = (prod.rename_axis('unit_id')
                 .reset_index()
                 .melt(id_vars='unit_id', var_name='component', value_name='score_prod'))

chk = (scores_long.query("subset == 'full'")
       .merge(prod_long, on=['unit_id', 'component'], how='inner'))
print(f"shared unit x component pairs: {len(chk)} "
      f"(this notebook: {(scores_long['subset'] == 'full').sum()}, production: {len(prod_long)})")
print(f"Pearson(full, production) = {chk['score'].corr(chk['score_prod']):.6f}")
print(f"max |difference| = {(chk['score'] - chk['score_prod']).abs().max():.3e}")

shared unit x component pairs: 102414 (this notebook: 102414, production: 102960)
Pearson(full, production) = 1.000000
max |difference| = 4.441e-16


## Write outputs


In [11]:
import os
os.makedirs(OUT_DIR, exist_ok=True)

scores_long.to_csv(f"{OUT_DIR}/spca_waggr_countSubset_scores_long.csv.gz", index=False)
subset_qc.to_csv(f"{OUT_DIR}/spca_waggr_countSubset_qc.csv", index=False)

print(f"wrote {OUT_DIR}/spca_waggr_countSubset_scores_long.csv.gz")
print(f"wrote {OUT_DIR}/spca_waggr_countSubset_qc.csv")

wrote /data1/rudenska/EYW/git_projects/SIG13/analysis_outs/qc_barcode_counts/spca_waggr_countSubset_scores_long.csv.gz
wrote /data1/rudenska/EYW/git_projects/SIG13/analysis_outs/qc_barcode_counts/spca_waggr_countSubset_qc.csv
